## Preprocessing

In [64]:
import pandas as pd
import numpy as np

In [65]:
df = pd.read_csv('data/preprocessing_v2.csv')
df.shape
df.head(1)

(9000, 8)

,engine_size,horsepower,mileage,age,weight,brand,fuel_type,price
0,2.30118,177.658371,87461.062098,4.118781,1542.567484,Honda,Hybrid,33795.453637


In [12]:
df.loc[df['fuel_type'] == 'Petrol', 'weight'].mean() - \
    df.loc[df['fuel_type'] == 'Diesel', 'weight'].mean()

np.float64(9.024435414894924)

In [70]:
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(exclude='number').columns.tolist()

## Model building

In [13]:
df = pd.read_csv('data/Model_Building_v3.csv')
df.shape
df.head(1)

(9000, 11)

,engine_size,horsepower,mileage,age,weight,fuel_type,brand_BMW,brand_Ford,brand_Honda,brand_Toyota,price
0,-0.254695,-1.721421,0.192759,0.398781,0.670278,2.0,0,0,1,0,-1959.092686


In [16]:
X = df.drop('price', axis=1)
y = df['price']

X.shape
y.shape

(9000, 10)

(9000,)

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

X_train.shape
X_test.shape
y_train.shape
y_test.shape

(7200, 10)

(1800, 10)

(7200,)

(1800,)

In [22]:
from sklearn.linear_model import Ridge
from sklearn.metrics import explained_variance_score

model = Ridge(alpha=0.02)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
explained_variance_score(y_test, y_pred)

Ridge(alpha=0.02)

0.9917535101016591

In [28]:
hp_index = X_train.columns.tolist().index('horsepower')
model.coef_[hp_index]

np.float64(129.65647478672585)

In [31]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import root_mean_squared_error

model = SGDRegressor(
    loss="squared_error",
    penalty="l2",
    alpha=0.0005,
    max_iter=2000,
    learning_rate='adaptive',
    random_state=0
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

root_mean_squared_error(y_test, y_pred)

SGDRegressor(alpha=0.0005, learning_rate='adaptive', max_iter=2000,
             random_state=0)

500.33389114374154

In [33]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

X_train_scaled.shape
X_train_poly.shape

(7200, 10)

(7200, 65)

In [37]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_poly, y_train)
y_pred = model.predict(X_test_poly)

LinearRegression()

In [38]:
from sklearn.metrics import r2_score

r2_score(y_test, y_pred)

0.9917047946749286

In [43]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(max_depth=6, min_samples_split=11, random_state=0)
tree.fit(X_train, y_train)
tree.tree_.node_count

DecisionTreeRegressor(max_depth=6, min_samples_split=11, random_state=0)

127

In [47]:
pd.Series(tree.feature_importances_, X_train.columns).sort_values()

horsepower      0.00000
mileage         0.00000
fuel_type       0.00000
weight          0.00000
brand_BMW       0.00000
brand_Ford      0.00000
brand_Toyota    0.00000
brand_Honda     0.00000
age             0.00013
engine_size     0.99987
dtype: float64

In [50]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

model = GradientBoostingRegressor(random_state=0)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
mean_absolute_error(y_train, y_train_pred)

GradientBoostingRegressor(random_state=0)

375.8003366975706

In [54]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.0001, 0.001]
}

grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=0),
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)

grid_search.fit(X_train, y_train)
grid_search.best_params_

GridSearchCV(cv=5, estimator=GradientBoostingRegressor(random_state=0),
             param_grid={'learning_rate': [0.0001, 0.001],
                         'n_estimators': [50, 100, 150]},
             scoring='neg_mean_squared_error')

{'learning_rate': 0.001, 'n_estimators': 150}

In [57]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

mlp = MLPRegressor(
    hidden_layer_sizes=(50, 32, 16),
    activation='tanh',
    solver='lbfgs',
    random_state=0,
    max_iter=5000
)

mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)

mean_squared_error(y_test, y_pred)

MLPRegressor(activation='tanh', hidden_layer_sizes=(50, 32, 16), max_iter=5000,
             random_state=0, solver='lbfgs')

5482060.330368006

In [62]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score

k_values = [5, 7, 11, 14, 20]
k_scores = {}

for k in k_values:
  knn = KNeighborsRegressor(n_neighbors=k)
  scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='r2')
  k_scores[k] = scores.mean()

best_k_score = max(k_scores, key=k_scores.get)

best_k_score

5

In [63]:
knn = KNeighborsRegressor()

param_grid = {'n_neighbors': [5, 7, 11, 14, 20]}

cv = GridSearchCV(knn, param_grid, cv=5, scoring='r2')
cv.fit(X_train, y_train)
cv.best_params_

GridSearchCV(cv=5, estimator=KNeighborsRegressor(),
             param_grid={'n_neighbors': [5, 7, 11, 14, 20]}, scoring='r2')

{'n_neighbors': 5}